# Classical-ML separability on the 104 model features (4-CSV real dataset)

**Fully self-contained.** The only thing you set is the folder with the 4 CSVs
(`car.csv`, `human.csv`, `car_nothing.csv`, `human_nothing.csv`) -- see `CSV_DIR` in Cell 1.
The feature-extraction code and the 104-feature list + normalization are **baked into this
notebook**; no other project files are required.

Question: can the **104 features the SNN uses** separate **human / car / nothing** with classical
models, and how much of that accuracy is genuine class learning vs **session identity**?

**Protocols.** A random window 5-fold (leaky upper bound) | B contiguous-block 5-fold (the SNN's
protocol) | C temporal holdout 70/30 (most session-disjoint) | D session probe (distinguish the
two *nothing* recordings, same label -> pure session signal). All graphs are in the final cell.

In [ ]:
# Cell 1: Config & imports  (set CSV_DIR -> everything else is self-contained)
import sys, subprocess
for pkg, imp in [('numpy','numpy'), ('pandas','pandas'), ('scipy','scipy'),
                 ('PyWavelets','pywt'), ('scikit-learn','sklearn'), ('matplotlib','matplotlib')]:
    try: __import__(imp)
    except ImportError:
        try: subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', pkg])
        except Exception as e: print('pip note:', pkg, e)

import os, time, warnings
import numpy as np, pandas as pd
warnings.filterwarnings('ignore')
import matplotlib.pyplot as plt
from sklearn.base import clone
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score, roc_auc_score, confusion_matrix

# ==================== SET YOUR CSV FOLDER HERE ====================
# Folder with the 4 recordings: car.csv, human.csv, car_nothing.csv, human_nothing.csv
# (each a column named 'amplitude', sampled at 1 kHz). Path is relative to this notebook.
# On a new machine this is the ONLY line you change.
CSV_DIR = "Goephone-Project/geophone_data"
# =================================================================

SCALE, SCENE, HOP, K = 25.4, 30000, 1500, 5   # amplitude calib, 30 s scenes, 1.5 s hop, 5 folds
REALS = {'car.csv': 'vehicle', 'human.csv': 'human', 'car_nothing.csv': 'nothing', 'human_nothing.csv': 'nothing'}
LABS = ['human', 'vehicle', 'nothing']
_missing = [f for f in REALS if not os.path.exists(os.path.join(CSV_DIR, f))]
assert not _missing, f"CSV_DIR={CSV_DIR!r} is missing {_missing}. Point CSV_DIR at the folder with the 4 CSVs."
print('CSV_DIR OK ->', os.path.abspath(CSV_DIR))

In [ ]:
# Cell 2: Feature bank -- BAKED IN verbatim from simgeo/features.py
# Defines FEATURE_NAMES (132), NFEAT, FS, NW, scene_precompute(), window_features().
# (No external file needed; this is the whole extractor.)

"""Feature bank v1 — 102 per-window features for the full-corpus screening pass.

Groups (order = FEATURE_NAMES order):
  LEGACY32   the notebook's 32 verbatim (site-tuned bands kept deliberately so their
             cross-terrain transfer is MEASURED, not asserted; band filters applied
             scene-wide then sliced — identical except edge transients)
  PHYS7      physics-band energies/fractions (wind 1-5, vehicle 5-25, footstep 20-90,
             high 90-180 Hz) — label.py band convention; fractions are distance-robust
  CAD12      cadence/gait modulation domain (envelope spectrum 0.3-12 Hz): cadence freq
             + salience, beat structure 2f0/4f0 (biped-vs-quadruped, Park & Dibazar),
             footfall rate, modulation-peak count + envelope entropy (single-vs-multiple
             candidates), inter-impulse intervals, gust AM
  ENG8       tonality/engine: harmonic comb 25-58 Hz orders 1-3, line count/stability,
             12-15 Hz wheel hop, engine-band beat depth (convoy cue), site-line monitor
             (44-47/50/55/100/150 Hz EXCLUDED from comb/lines, measured separately)
  WX2        weather gates: rain-band impulse rate, high-band kurtosis
  CEP16      mfcc_1..8 + lfcc_1..8 (24-filter banks, 1-200 Hz; mel ~ linear down here)
  WPE16      db4 wavelet-packet level-4 leaf energy fractions
  AR8        AR(8) coefficients (Yule-Walker/Levinson on the window autocorrelation)
  NL1        Higuchi fractal dimension (kmax=8)

Window = 3.0 s / 3000 samples @ 1 kHz (matches the corpus label windows).
"""
import numpy as np
from scipy import signal as sig
from scipy import stats as spstats
from scipy.fft import dct
import pywt

FS = 1000.0
NW = 3000
NFFT_ENV = 8192                       # envelope spectrum zero-pad (0.122 Hz resolution)

# ---------------------------------------------------------------- filter banks
LEGACY_BANDS = {
    "LOW_FREQ": (20, 30), "CAR_APPROACH": (30, 34), "CAR_PEAK": (34, 40),
    "CAR_TAIL": (40, 48), "MID_GAP": (48, 60), "HUMAN_PEAK": (60, 70),
    "HUMAN_TAIL": (70, 80), "HIGH_FREQ": (90, 100),
}
PHYS_BANDS = {"wind": (1, 5), "veh": (5, 25), "foot": (20, 90),
              "high": (90, 180), "hop": (12, 15), "eng": (20, 60), "rain": (60, 200)}


def _bp(lo, hi, order=4):
    nyq = FS / 2
    return sig.butter(order, [max(lo / nyq, 1e-3), min(hi / nyq, 0.999)],
                      btype="band", output="sos")


_SOS_LEG = {k: _bp(lo, hi) for k, (lo, hi) in LEGACY_BANDS.items()}
_SOS_PHY = {k: _bp(lo, hi) for k, (lo, hi) in PHYS_BANDS.items()}
_SOS_ENVLP = sig.butter(4, 20 / (FS / 2), btype="low", output="sos")

# rfft grids
_FREQS = np.fft.rfftfreq(NW, 1 / FS)                 # 0.333 Hz bins
_FM = np.fft.rfftfreq(NFFT_ENV, 1 / FS)              # envelope-spectrum grid
_M_MOD = (_FM >= 0.3) & (_FM <= 12.0)                # modulation analysis band
_M_CAD = (_FM >= 0.8) & (_FM <= 4.0)                 # human cadence search
_M_FFR = (_FM >= 0.8) & (_FM <= 10.0)                # footfall-rate search (quadruped 4-8)
_M_PKS = (_FM >= 0.5) & (_FM <= 8.0)                 # multi-subject peak count
_M_GUST = (_FM >= 0.05) & (_FM <= 0.5)               # gust AM
_M_GALL = (_FM >= 0.05) & (_FM <= 12.0)

# site/mains tonal lines (rig-measured 44-47/55 + optional 50 Hz family) — these are
# EXCLUDED from comb/line features and monitored separately as line_mains
SITE_LINES = np.array([44, 45, 46, 47, 50, 55, 100, 150], float)
_SITE_BIN = np.zeros(len(_FREQS), bool)
for _f in SITE_LINES:
    _SITE_BIN |= np.abs(_FREQS - _f) <= 0.75
_M_LINES = (_FREQS >= 10) & (_FREQS <= 180)
# engine comb f0 candidates: 25-58 Hz bins, excluding site-line bins
_COMB_K = np.where((_FREQS >= 25) & (_FREQS <= 58) & ~_SITE_BIN)[0]

# ---- v2 additional grids (new feature groups) ----
_M_MOD1_3 = (_FM >= 1) & (_FM <= 3)        # human cadence band (modulation)
_M_MOD3_8 = (_FM >= 3) & (_FM <= 8)        # quadruped / fast-footfall band
_M_MOD8_12 = (_FM >= 8) & (_FM <= 12)      # herd/group blur band
_ROTOR_K = np.where((_FREQS >= 12) & (_FREQS <= 22))[0]   # helicopter main-rotor BPF comb
_PUMP = (_FREQS >= 75) & (_FREQS <= 180)                  # irrigation-pump tonal band


def _tri_fb(edges, freqs):
    """Triangular filterbank (nfilt x nbins) from band edges."""
    nf = len(edges) - 2
    fb = np.zeros((nf, len(freqs)))
    for i in range(nf):
        l, c, r = edges[i], edges[i + 1], edges[i + 2]
        fb[i] = np.clip(np.minimum((freqs - l) / max(c - l, 1e-9),
                                   (r - freqs) / max(r - c, 1e-9)), 0, None)
    return fb


_mel = lambda f: 2595 * np.log10(1 + f / 700)
_imel = lambda m: 700 * (10 ** (m / 2595) - 1)
_FB_MEL = _tri_fb(_imel(np.linspace(_mel(1), _mel(200), 26)), _FREQS)
_FB_LOG = _tri_fb(np.geomspace(2, 200, 26), _FREQS)

FEATURE_NAMES = (
    # LEGACY32
    ["energy_low_freq", "energy_car_approach", "energy_car_peak", "energy_car_tail",
     "energy_mid_gap", "energy_human_peak", "energy_human_tail", "energy_high_freq",
     "rms_total", "peak_to_peak", "variance", "zcr", "kurtosis", "skewness",
     "spectral_centroid", "spectral_bandwidth", "spectral_rolloff", "spectral_flatness",
     "spectral_entropy", "dominant_freq", "sta_lta_ratio", "event_count",
     "mean_burst_len", "activity_concentration", "temporal_entropy", "burst_efficiency",
     "autocorr_250", "autocorr_500", "ratio_car_human", "ratio_human_car",
     "centroid_car_band", "centroid_human_band"]
    # PHYS7
    + ["log_rms", "frac_wind_1_5", "frac_veh_5_25", "frac_foot_20_90",
       "frac_high_90_180", "centroid_foot_band", "ratio_veh_foot"]
    # CAD12
    + ["cad_freq", "cad_salience", "cad_harm2", "cad_harm4", "footfall_rate",
       "mod_peak_count", "env_entropy", "env_cv", "env_ac_strength",
       "iii_mean", "iii_cv", "gust_mod"]
    # ENG8
    + ["comb_salience", "comb_f0", "n_lines", "line_stability", "hop_band_frac",
       "beat_depth", "line_mains", "tonality_max"]
    # WX2
    + ["rain_impulse_rate", "highband_kurtosis"]
    # CEP16
    + [f"mfcc_{i}" for i in range(1, 9)] + [f"lfcc_{i}" for i in range(1, 9)]
    # WPE16
    + [f"wpe_{i}" for i in range(16)]
    # AR8
    + [f"ar_{i}" for i in range(1, 9)]
    # NL1
    + ["higuchi_fd"]
    # ==== v2 NEW (30) ====
    # MOD2 — modulation-spectrum refinement (single/multi + biped/quad/vehicle)
    + ["mod_e_1_3", "mod_e_3_8", "mod_e_8_12", "mod_ratio_low_mid", "mod_peak2_ratio",
       "mod_flatness", "mod_centroid"]
    # III2 — inter-impulse-interval stats (single/multi)
    + ["impulse_density", "iii_entropy", "iii_skew", "iii_range_norm", "impulse_amp_cv"]
    # TONAL — tonal-vs-impulsive / machinery+aircraft confuser separation (nothing/class)
    + ["spectral_crest", "n_sharp_lines", "max_line_prom", "tonal_index",
       "heli_comb_salience", "pump_band_frac", "tonal_in_foot"]
    # STAT2 — stationarity / temporal structure (between-class)
    + ["centroid_var", "rms_cv_sub", "spectral_flux", "temporal_centroid", "attack_sharpness"]
    # HOS — higher-order spectral (engine harmonic coupling)
    + ["spec_skew", "spec_kurt", "comb_stability"]
    # COOC — co-occurrence / multi-source complexity (mixed scenes)
    + ["band_occupancy", "spectral_entropy_full", "veh_foot_simultaneity"]
)
NFEAT = len(FEATURE_NAMES)
assert NFEAT == 132, NFEAT


# ------------------------------------------------------------ scene precompute
def scene_precompute(x):
    """One-time per-scene work: band-filtered signals, envelopes, impulse picks."""
    x = np.asarray(x, np.float64)
    pre = {"x": x}
    for k, sos in _SOS_LEG.items():
        pre["leg_" + k] = sig.sosfilt(sos, x)
    for k, sos in _SOS_PHY.items():
        pre["phy_" + k] = sig.sosfilt(sos, x)
    # broadband envelope (|x| lowpassed 20 Hz) for cadence/modulation work
    pre["env"] = np.clip(sig.sosfilt(_SOS_ENVLP, np.abs(x)), 0, None)
    pre["env_eng"] = np.clip(sig.sosfilt(_SOS_ENVLP, np.abs(pre["phy_eng"])), 0, None)
    env_rain = np.clip(sig.sosfilt(_SOS_ENVLP, np.abs(pre["phy_rain"])), 0, None)
    # impulse picks (scene-wide; windows subselect)
    med = np.median(pre["env"]) + 1e-12
    pk, _ = sig.find_peaks(pre["env"], height=2 * med, distance=int(0.08 * FS))
    pre["imp_t"] = pk
    medr = np.median(env_rain) + 1e-12
    pkr, _ = sig.find_peaks(env_rain, height=3 * medr, distance=int(0.02 * FS))
    pre["rain_t"] = pkr
    return pre


# ------------------------------------------------------------ helpers
def _higuchi(w, kmax=8):
    n = len(w)
    lk = np.empty(kmax)
    for k in range(1, kmax + 1):
        Lm = 0.0
        for m in range(k):
            idx = np.arange(m, n, k)
            if len(idx) < 2:
                continue
            Lm += np.abs(np.diff(w[idx])).sum() * (n - 1) / (len(idx) - 1) / k
        lk[k - 1] = Lm / k + 1e-30
    kk = np.log(1.0 / np.arange(1, kmax + 1))
    return float(np.polyfit(kk, np.log(lk), 1)[0])


def _levinson(r, p=8):
    a = np.zeros(p + 1); a[0] = 1.0; e = r[0] + 1e-30
    for i in range(1, p + 1):
        k = -(r[i] + a[1:i] @ r[i - 1:0:-1]) / e
        a[1:i + 1] = a[1:i + 1] + k * a[i - 1::-1][:i]
        e *= (1 - k * k) + 1e-30
    return a[1:]


# ------------------------------------------------------------ per-window
def window_features(pre, i0):
    f = np.empty(NFEAT, np.float64)
    j = 0
    w = pre["x"][i0:i0 + NW]
    aw = np.abs(w)

    # ---- LEGACY32 -----------------------------------------------------------
    band_e = {}
    for k in LEGACY_BANDS:
        e = float(np.sqrt(np.mean(pre["leg_" + k][i0:i0 + NW] ** 2)))
        band_e[k] = e; f[j] = e; j += 1
    rms_total = float(np.sqrt(np.mean(w ** 2)))
    f[j] = rms_total; j += 1
    f[j] = float(np.ptp(w)); j += 1
    f[j] = float(np.var(w)); j += 1
    f[j] = np.sum(np.diff(np.sign(w)) != 0) / NW; j += 1
    f[j] = float(spstats.kurtosis(w)); j += 1
    f[j] = float(spstats.skew(w)); j += 1
    mag = np.abs(np.fft.rfft(w))
    tot_mag = mag.sum() + 1e-10
    psd = mag ** 2
    tot_psd = psd.sum() + 1e-10
    sc = float((_FREQS * mag).sum() / tot_mag)
    f[j] = sc; j += 1
    f[j] = float(np.sqrt((((_FREQS - sc) ** 2) * mag).sum() / tot_mag)); j += 1
    cs = np.cumsum(psd)
    f[j] = _FREQS[min(np.searchsorted(cs, 0.95 * tot_psd), len(_FREQS) - 1)]; j += 1
    f[j] = float(np.exp(np.mean(np.log(psd + 1e-10))) / (psd.mean() + 1e-10)); j += 1
    pn = psd / tot_psd
    f[j] = float(-(pn * np.log2(pn + 1e-10)).sum()); j += 1
    f[j] = _FREQS[int(np.argmax(mag))]; j += 1
    lta = aw.mean() + 1e-10
    sta = np.convolve(aw, np.ones(100) / 100, mode="valid")
    f[j] = float(sta.max() / lta); j += 1
    pk, _ = sig.find_peaks(aw, height=2 * rms_total)
    f[j] = float(len(pk)); j += 1
    above = aw > rms_total
    d = np.diff(above.astype(np.int8))
    st, en = np.where(d == 1)[0], np.where(d == -1)[0]
    if len(st) and len(en):
        if en[0] < st[0]:
            en = en[1:]
        nb = min(len(st), len(en))
        f[j] = float(np.mean(en[:nb] - st[:nb])) if nb else 0.0
    else:
        f[j] = 0.0
    j += 1
    se = np.sort(w ** 2)[::-1]
    f[j] = float(se[:NW // 4].sum() / (se.sum() + 1e-10)); j += 1
    hist, _ = np.histogram(w, bins=50, density=True)
    hist = hist[hist > 0]; hn = hist / (hist.sum() + 1e-10)
    f[j] = float(-(hn * np.log2(hn + 1e-10)).sum()); j += 1
    f[j] = float((w[above] ** 2).sum() / ((w ** 2).sum() + 1e-10)); j += 1
    for lag in (250, 500):
        ac = np.corrcoef(w[:-lag], w[lag:])[0, 1]
        f[j] = float(ac) if np.isfinite(ac) else 0.0
        j += 1
    car_e = band_e["CAR_PEAK"] + 1e-10
    hum_e = band_e["HUMAN_PEAK"] + 1e-10
    f[j] = car_e / hum_e; j += 1
    f[j] = hum_e / car_e; j += 1
    for lo, hi in ((34, 48), (60, 80)):
        m = (_FREQS >= lo) & (_FREQS <= hi)
        f[j] = float((_FREQS[m] * mag[m]).sum() / (mag[m].sum() + 1e-10)); j += 1

    # ---- PHYS7 --------------------------------------------------------------
    f[j] = float(np.log10(rms_total + 1e-12)); j += 1
    phys_rms = {}
    for k in ("wind", "veh", "foot", "high"):
        phys_rms[k] = float(np.sqrt(np.mean(pre["phy_" + k][i0:i0 + NW] ** 2)))
        f[j] = phys_rms[k] / (rms_total + 1e-12); j += 1
    m = (_FREQS >= 20) & (_FREQS <= 90)
    f[j] = float((_FREQS[m] * mag[m]).sum() / (mag[m].sum() + 1e-10)); j += 1
    f[j] = phys_rms["veh"] / (phys_rms["foot"] + 1e-12); j += 1

    # ---- CAD12 --------------------------------------------------------------
    e = pre["env"][i0:i0 + NW]
    e0 = e - e.mean()
    Espec = np.abs(np.fft.rfft(e0, NFFT_ENV)) ** 2
    modmed = np.median(Espec[_M_MOD]) + 1e-20
    kc = np.where(_M_CAD)[0]
    kpk = kc[int(np.argmax(Espec[kc]))]
    f0 = _FM[kpk]
    f[j] = float(f0); j += 1
    f[j] = float(Espec[kpk] / modmed); j += 1
    pf0 = Espec[max(kpk - 2, 0):kpk + 3].sum() + 1e-20
    for mult in (2, 4):
        kh = int(round(kpk * mult))
        ph = Espec[max(kh - 2, 0):kh + 3].sum() if kh + 3 <= len(Espec) else 0.0
        f[j] = float(ph / pf0); j += 1
    kr = np.where(_M_FFR)[0]
    f[j] = float(_FM[kr[int(np.argmax(Espec[kr]))]]); j += 1
    kp = np.where(_M_PKS)[0]
    pks, _ = sig.find_peaks(Espec[kp], height=3 * modmed)
    f[j] = float(len(pks)); j += 1
    Em = Espec[_M_MOD] / (Espec[_M_MOD].sum() + 1e-20)
    f[j] = float(-(Em * np.log2(Em + 1e-20)).sum()); j += 1
    f[j] = float(e.std() / (e.mean() + 1e-12)); j += 1
    ac = np.fft.irfft(Espec)                      # envelope autocorrelation (padded->linear)
    f[j] = float(np.max(ac[250:1250]) / (ac[0] + 1e-20)); j += 1
    it = pre["imp_t"]
    it = it[(it >= i0) & (it < i0 + NW)]
    if len(it) >= 3:
        iii = np.diff(it) / FS
        f[j] = float(iii.mean()); j += 1
        f[j] = float(iii.std() / (iii.mean() + 1e-9)); j += 1
    else:
        f[j] = 0.0; j += 1
        f[j] = 0.0; j += 1
    f[j] = float(Espec[_M_GUST].sum() / (Espec[_M_GALL].sum() + 1e-20)); j += 1

    # ---- ENG8 ---------------------------------------------------------------
    locmed = sig.medfilt(psd, 51) + 1e-12
    pnorm = psd / locmed
    best_s, best_k = 0.0, _COMB_K[0]
    for k in _COMB_K:
        s = 0.0; nv = 0
        for o in (1, 2, 3):
            ko = k * o
            if ko + 1 < len(pnorm) and not _SITE_BIN[ko - 1:ko + 2].any():
                s += pnorm[ko - 1:ko + 2].max(); nv += 1   # skip orders on site/mains lines
        if nv >= 2:                                        # need >=2 clean orders (real comb)
            s /= nv
            if s > best_s:
                best_s, best_k = s, k
    f[j] = float(best_s); j += 1
    f[j] = float(_FREQS[best_k]); j += 1
    lpk, _ = sig.find_peaks(np.where(_M_LINES & ~_SITE_BIN, pnorm, 0.0), height=6.0)
    f[j] = float(len(lpk)); j += 1
    sub = np.stack([np.log10(np.abs(np.fft.rfft(w[s * 1000:(s + 1) * 1000])) ** 2 + 1e-12)
                    for s in range(3)])
    fsub = np.fft.rfftfreq(1000, 1 / FS)
    msub = (fsub >= 10) & (fsub <= 100)
    cc = np.corrcoef(sub[:, msub])
    f[j] = float((cc[0, 1] + cc[0, 2] + cc[1, 2]) / 3); j += 1
    f[j] = float(np.sqrt(np.mean(pre["phy_hop"][i0:i0 + NW] ** 2)) / (rms_total + 1e-12)); j += 1
    ee = pre["env_eng"][i0:i0 + NW]
    f[j] = float(ee.std() / (ee.mean() + 1e-12)); j += 1
    f[j] = float(pnorm[_SITE_BIN].mean()); j += 1
    f[j] = float(pnorm[_M_LINES].max()); j += 1

    # ---- WX2 ----------------------------------------------------------------
    rt = pre["rain_t"]
    f[j] = float(((rt >= i0) & (rt < i0 + NW)).sum() / (NW / FS)); j += 1
    f[j] = float(spstats.kurtosis(pre["phy_rain"][i0:i0 + NW])); j += 1

    # ---- CEP16 --------------------------------------------------------------
    for fb in (_FB_MEL, _FB_LOG):
        c = dct(np.log(fb @ mag + 1e-10), type=2, norm="ortho")
        f[j:j + 8] = c[1:9]; j += 8

    # ---- WPE16 --------------------------------------------------------------
    wp = pywt.WaveletPacket(w, "db4", maxlevel=4)
    en = np.array([float((n.data ** 2).sum()) for n in wp.get_level(4, "natural")])
    f[j:j + 16] = en / (en.sum() + 1e-20); j += 16

    # ---- AR8 ----------------------------------------------------------------
    r = np.fft.irfft(psd)[:9]
    f[j:j + 8] = _levinson(r, 8); j += 8

    # ---- NL1 ----------------------------------------------------------------
    f[j] = _higuchi(w); j += 1

    # ==== v2 NEW (reuse mag/psd/Espec/modmed/it/pnorm/phys_rms/sub/msub/fsub) ====
    # MOD2 — modulation-spectrum refinement
    Emod = Espec[_M_MOD]; Emt = Emod.sum() + 1e-20
    f[j] = float(Espec[_M_MOD1_3].sum() / Emt); j += 1                 # mod_e_1_3 (human cadence)
    f[j] = float(Espec[_M_MOD3_8].sum() / Emt); j += 1                 # mod_e_3_8 (quadruped/fast)
    f[j] = float(Espec[_M_MOD8_12].sum() / Emt); j += 1                # mod_e_8_12 (group blur)
    f[j] = float(Espec[_M_MOD1_3].sum() / (Espec[_M_MOD3_8].sum() + 1e-20)); j += 1  # mod_ratio_low_mid
    mpk, _ = sig.find_peaks(Espec[_M_PKS], height=3 * modmed)
    if len(mpk) >= 2:
        top = np.sort(Espec[_M_PKS][mpk])[::-1]; f[j] = float(top[1] / (top[0] + 1e-20))
    else:
        f[j] = 0.0
    j += 1                                                             # mod_peak2_ratio (multi)
    f[j] = float(np.exp(np.mean(np.log(Emod + 1e-20))) / (Emod.mean() + 1e-20)); j += 1  # mod_flatness
    f[j] = float((_FM[_M_MOD] * Emod).sum() / Emt); j += 1             # mod_centroid (rhythm speed)

    # III2 — inter-impulse-interval stats (single vs multiple)
    f[j] = float(len(it) / (NW / FS)); j += 1                          # impulse_density
    if len(it) >= 4:
        iii = np.diff(it) / FS
        h, _ = np.histogram(iii, bins=8); h = h[h > 0] / (h.sum() + 1e-12)
        f[j] = float(-(h * np.log2(h + 1e-12)).sum()); j += 1          # iii_entropy
        f[j] = float(spstats.skew(iii)); j += 1                        # iii_skew
        f[j] = float((np.percentile(iii, 90) - np.percentile(iii, 10)) / (np.median(iii) + 1e-9)); j += 1  # iii_range_norm
        amps = pre["env"][it]; f[j] = float(amps.std() / (amps.mean() + 1e-12)); j += 1  # impulse_amp_cv
    else:
        f[j] = 0.0; j += 1; f[j] = 0.0; j += 1; f[j] = 0.0; j += 1; f[j] = 0.0; j += 1

    # TONAL — tonal-vs-impulsive / machinery+aircraft confuser separation
    f[j] = float(psd.max() / (psd.mean() + 1e-20)); j += 1             # spectral_crest
    ratio_line = np.where((_FREQS >= 10) & (_FREQS <= 200) & ~_SITE_BIN, pnorm, 0.0)
    slp, _ = sig.find_peaks(ratio_line, height=8.0)
    f[j] = float(len(slp)); j += 1                                     # n_sharp_lines (non-mains)
    f[j] = float(ratio_line.max()); j += 1                             # max_line_prom
    inb = psd[(_FREQS >= 10) & (_FREQS <= 200)].sum() + 1e-20
    f[j] = float(psd[ratio_line > 8.0].sum() / inb); j += 1            # tonal_index
    best_h = 0.0
    for k in _ROTOR_K:                                                 # helicopter rotor-BPF comb 12-22 Hz
        s = 0.0; nv = 0
        for o in (1, 2, 3):
            ko = k * o
            if ko + 1 < len(pnorm) and not _SITE_BIN[ko - 1:ko + 2].any():
                s += pnorm[ko - 1:ko + 2].max(); nv += 1
        if nv >= 2:
            best_h = max(best_h, s / nv)
    f[j] = float(best_h); j += 1                                       # heli_comb_salience
    f[j] = float(psd[_PUMP].sum() / tot_psd); j += 1                   # pump_band_frac (75-180)
    fbm = (_FREQS >= 20) & (_FREQS <= 90)
    f[j] = float(psd[(ratio_line > 8.0) & fbm].sum() / (psd[fbm].sum() + 1e-20)); j += 1  # tonal_in_foot

    # STAT2 — stationarity / temporal structure
    subc = []; subr = []
    for s_ in range(3):
        ws = w[s_ * 1000:(s_ + 1) * 1000]; m_ = np.abs(np.fft.rfft(ws))
        subc.append((fsub * m_).sum() / (m_.sum() + 1e-10)); subr.append(np.sqrt(np.mean(ws ** 2)))
    subc = np.array(subc); subr = np.array(subr)
    f[j] = float(subc.std()); j += 1                                   # centroid_var (non-stationary)
    f[j] = float(subr.std() / (subr.mean() + 1e-12)); j += 1           # rms_cv_sub (burstiness)
    f[j] = float(np.mean(np.abs(np.diff(sub, axis=0)))); j += 1        # spectral_flux
    w2 = w ** 2; f[j] = float((np.arange(NW) * w2).sum() / ((w2.sum() + 1e-20) * NW)); j += 1  # temporal_centroid
    if len(it) >= 2:
        sl = [(pre["env"][ti] - pre["env"][max(ti - 10, 0)]) / 0.01 for ti in it[:20]]
        f[j] = float(np.mean(sl)) if sl else 0.0
    else:
        f[j] = 0.0
    j += 1                                                             # attack_sharpness

    # HOS — higher-order spectral
    f[j] = float(spstats.skew(psd)); j += 1                            # spec_skew
    f[j] = float(spstats.kurtosis(psd)); j += 1                        # spec_kurt (tonal peaky)
    f[j] = float(np.mean(np.std(sub[:, msub], axis=0))); j += 1        # comb_stability (engine phase-lock)

    # COOC — co-occurrence / multi-source complexity (mixed scenes)
    f[j] = float(sum(1 for k in ("wind", "veh", "foot", "high")
                     if phys_rms.get(k, 0) / (rms_total + 1e-12) > 0.15)); j += 1  # band_occupancy
    pnf = psd / tot_psd; f[j] = float(-(pnf * np.log2(pnf + 1e-12)).sum()); j += 1  # spectral_entropy_full
    f[j] = float(phys_rms.get("veh", 0) / (rms_total + 1e-12) * min(len(it) / (NW / FS), 5.0)); j += 1  # veh_foot_simultaneity

    assert j == NFEAT
    return f


In [ ]:
# Cell 3: The 104 features the model uses + its normalization -- BAKED IN from scaler.json
FEATURES = ['beat_depth', 'env_cv', 'sta_lta_ratio', 'energy_low_freq', 'highband_kurtosis', 'n_sharp_lines', 'kurtosis', 'spectral_centroid', 'wpe_9', 'lfcc_2', 'temporal_entropy', 'energy_mid_gap', 'env_ac_strength', 'energy_human_peak', 'wpe_13', 'spectral_bandwidth', 'env_entropy', 'impulse_amp_cv', 'footfall_rate', 'rms_cv_sub', 'cad_freq', 'rain_impulse_rate', 'energy_human_tail', 'mfcc_4', 'comb_salience', 'attack_sharpness', 'cad_salience', 'mod_ratio_low_mid', 'gust_mod', 'event_count', 'mod_e_1_3', 'zcr', 'mod_e_3_8', 'mod_centroid', 'mod_e_8_12', 'burst_efficiency', 'frac_wind_1_5', 'tonality_max', 'ar_2', 'wpe_5', 'tonal_in_foot', 'energy_high_freq', 'hop_band_frac', 'impulse_density', 'tonal_index', 'iii_cv', 'mod_peak2_ratio', 'veh_foot_simultaneity', 'wpe_2', 'lfcc_3', 'lfcc_5', 'heli_comb_salience', 'iii_range_norm', 'iii_entropy', 'lfcc_4', 'iii_mean', 'frac_foot_20_90', 'band_occupancy', 'centroid_human_band', 'mod_peak_count', 'mfcc_3', 'mfcc_6', 'lfcc_7', 'lfcc_1', 'centroid_var', 'lfcc_6', 'iii_skew', 'ratio_car_human', 'dominant_freq', 'cad_harm2', 'comb_f0', 'skewness', 'temporal_centroid', 'energy_car_peak', 'wpe_8', 'peak_to_peak', 'activity_concentration', 'wpe_12', 'variance', 'log_rms', 'spectral_flatness', 'ar_1', 'wpe_11', 'wpe_10', 'mean_burst_len', 'wpe_14', 'wpe_15', 'spectral_rolloff', 'higuchi_fd', 'line_stability', 'wpe_6', 'frac_high_90_180', 'cad_harm4', 'wpe_7', 'comb_stability', 'spec_kurt', 'frac_veh_5_25', 'ratio_veh_foot', 'mfcc_8', 'centroid_foot_band', 'wpe_1', 'mfcc_2', 'spectral_crest', 'line_mains']
SCALER_MEAN = np.array([0.5985817909240723, 0.5319151878356934, 2.1318209171295166, 120748.796875, 4.09763240814209, 6.788379192352295, 3.32893705368042, 125.3554916381836, 0.0024644818622618914, -0.8665940165519714, 4.460330963134766, 99411.5, 0.2716398537158966, 61309.86328125, 0.008875771425664425, 94.5416488647461, 5.586612701416016, 0.1627245396375656, 3.704439163208008, 0.10194352269172668, 2.240373134613037, 1.9976617097854614, 70831.359375, -0.5335564613342285, 131.2105255126953, 11231048.0, 15.94491958618164, 0.7215620279312134, 0.08665671944618225, 69.09039306640625, 0.2336898297071457, 0.18735021352767944, 0.4250020682811737, 5.055960178375244, 0.23132675886154175, 0.8376591205596924, 0.09354393929243088, 655.6851196289062, 1.062522530555725, 0.019779987633228302, 0.1845870167016983, 72608.09375, 0.11513176560401917, 2.425184488296509, 0.24846485257148743, 0.28928789496421814, 0.6461299061775208, 0.6631930470466614, 0.10734641551971436, -0.32927408814430237, -0.001979377120733261, 48.6004753112793, 0.7406003475189209, 1.142194390296936, -0.616628885269165, 0.19520783424377441, 0.6308261156082153, 2.6867144107818604, 66.11698150634766, 4.970997333526611, -0.017188647761940956, -0.18937429785728455, -0.17214970290660858, -6.50899600982666, 4.372812747955322, -0.16557593643665314, 0.35853540897369385, 2.247318983078003, 65.34129333496094, 0.6316949129104614, 38.318965911865234, 0.07501006126403809, 0.5005367994308472, 70550.828125, 0.002205413533374667, 3738371.0, 0.7643737196922302, 0.01014467142522335, 917902559543296.0, 0.07199300080537796, 0.06762820482254028, -1.5563850402832031, 0.003271981840953231, 0.004646364599466324, 4.033365726470947, 0.0066765546798706055, 0.008954174816608429, 170.99957275390625, 0.5665373802185059, 0.5008951425552368, 0.11151789873838425, 0.4610508680343628, 0.4220869243144989, 0.058264024555683136, 0.39738547801971436, 461.92083740234375, 0.3072602152824402, 0.6003686785697937, 0.10672048479318619, 55.33070755004883, 0.21611946821212769, -0.8718761801719666, 237.58412170410156, 14.50287914276123], np.float32)
SCALER_STD  = np.array([0.38084927201271057, 0.38254469633102417, 1.225221037864685, 6399820.5, 10.846023559570312, 5.869551181793213, 9.24769401550293, 45.92799758911133, 0.006266511976718903, 2.448320150375366, 0.7354631423950195, 9823916.0, 0.16243606805801392, 3820615.5, 0.01937858574092388, 35.652462005615234, 0.5355439782142639, 0.2358327955007553, 2.451845407485962, 0.1067647859454155, 0.9693905711174011, 3.5135281085968018, 5305403.0, 1.1308757066726685, 3340.811279296875, 591900160.0, 28.534900665283203, 0.8310821652412415, 0.12464020401239395, 32.06660461425781, 0.1353714019060135, 0.09054193645715714, 0.14859658479690552, 1.2283244132995605, 0.12687477469444275, 0.042148407548666, 0.09767426550388336, 7794.40576171875, 1.2248899936676025, 0.037880148738622665, 0.24144846200942993, 4990536.5, 0.13091200590133667, 2.440107822418213, 0.26148566603660583, 0.29708331823349, 0.26035425066947937, 1.0696860551834106, 0.1225656196475029, 1.4573125839233398, 0.8868258595466614, 650.270751953125, 0.9961696267127991, 1.0547873973846436, 1.0802990198135376, 0.21582935750484467, 0.2567439079284668, 0.8974160552024841, 4.051968574523926, 2.0589370727539062, 1.3078103065490723, 0.6390200257301331, 0.5609237551689148, 2.883472204208374, 3.8935117721557617, 0.6425998210906982, 0.7282551527023315, 11.009016036987305, 57.22421646118164, 1.7668113708496094, 10.146232604980469, 0.4077649712562561, 0.06384554505348206, 3771585.25, 0.005679000169038773, 196864944.0, 0.09637527167797089, 0.02574186772108078, np.inf, 0.9497552514076233, 0.09473223984241486, 0.7182541489601135, 0.008430507965385914, 0.009988727048039436, 3.372288465499878, 0.012431662529706955, 0.017483564093708992, 102.63581848144531, 0.31952306628227234, 0.2648824453353882, 0.14586131274700165, 0.2927480936050415, 3.5886287689208984, 0.09308839589357376, 0.06452462822198868, 436.0224914550781, 0.26114514470100403, 0.8564202785491943, 0.5489252209663391, 9.651122093200684, 0.19646471738815308, 2.1589133739471436, 246.41802978515625, 79.03286743164062], np.float32)
SCALER_CLIP = 8.0
assert len(FEATURES) == len(SCALER_MEAN) == len(SCALER_STD)
fidx = [FEATURE_NAMES.index(f) for f in FEATURES]   # map the 104 into the 132-feature bank
def z(X): return np.clip((X - SCALER_MEAN) / SCALER_STD, -SCALER_CLIP, SCALER_CLIP)
print('using', len(FEATURES), 'features (subset of', NFEAT, 'computed)')

In [ ]:
# Cell 4: Load the 4 CSVs -> 104 window features (uses the baked-in extractor)
def feat(p):
    df = pd.read_csv(p)
    a = (df['amplitude'] if 'amplitude' in df.columns else df.iloc[:, -1]).to_numpy(np.float32) * SCALE
    fe = []
    for c0 in range(0, len(a), SCENE):
        seg = a[c0:c0 + SCENE]
        if len(seg) < NW: continue
        pre = scene_precompute(seg)
        for i0 in range(0, len(seg) - NW + 1, HOP): fe.append(window_features(pre, i0).astype(np.float32))
    return np.nan_to_num(np.stack(fe)) if fe else np.empty((0, NFEAT), np.float32)

t0 = time.time(); X, y, src = [], [], []
for fn, cls in REALS.items():
    Xr = feat(os.path.join(CSV_DIR, fn))[:, fidx]; n = len(Xr)
    X.append(Xr.astype(np.float32)); y += [cls] * n; src += [fn] * n
    print(f'  {fn:18s} {n:4d} windows -> {cls}')
X = np.vstack(X); y = np.array(y); src = np.array(src); Xz = z(X)
# contiguous time-block folds PER FILE (the SNN's split = time slices of one recording)
blk = np.zeros(len(y), int)
for fn in REALS:
    idx = np.where(src == fn)[0]; blk[idx] = np.arange(len(idx)) * K // max(len(idx), 1)
print(f'REAL windows: {len(y)}  per class: {dict((c, int((y==c).sum())) for c in LABS)}  ({time.time()-t0:.0f}s)')

In [ ]:
# Cell 5: Classical models + CV runner
models = {
    'tree_d2 (~3 leaves)': DecisionTreeClassifier(max_depth=2, random_state=0),
    'tree_d5':             DecisionTreeClassifier(max_depth=5, random_state=0),
    'rf200':               RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    'svm_rbf':             SVC(kernel='rbf', C=1.0, random_state=0),
    'logreg (linear)':     LogisticRegression(max_iter=3000),
}
def run(splits):
    out = {}
    for nm, clf in models.items():
        ac, ba = [], []
        for tr, te in splits:
            c = clone(clf).fit(Xz[tr], y[tr]); p = c.predict(Xz[te])
            ac.append(accuracy_score(y[te], p)); ba.append(balanced_accuracy_score(y[te], p))
        out[nm] = (float(np.mean(ac)), float(np.mean(ba)))
    return out
SNN_REF = 0.980  # SNN block-5-fold reference
def show(tag, res):
    print(tag)
    for nm, (a, b) in res.items(): print(f'  {nm:22s} acc={a:.3f}  bal={b:.3f}')

In [ ]:
# Cell 6: Protocol A - random window 5-fold  (leaky upper bound)
A_res = run(list(StratifiedKFold(5, shuffle=True, random_state=0).split(Xz, y)))
show('A) random window 5-fold (leaky):', A_res)

In [ ]:
# Cell 7: Protocol B - contiguous-block 5-fold  (the SNN's protocol)
B_res = run([(np.where(blk != k)[0], np.where(blk == k)[0]) for k in range(K)])
show('B) contiguous-block 5-fold (SNN protocol):', B_res)

In [ ]:
# Cell 8: Protocol C - temporal holdout 70/30  (most session-disjoint; 2-window gap kills overlap leak)
tr, te = [], []
for fn in REALS:
    idx = np.where(src == fn)[0]; cut = int(0.7 * len(idx))
    tr += list(idx[:max(cut - 1, 0)]); te += list(idx[cut + 1:])
C_res = run([(np.array(tr), np.array(te))])
show('C) temporal holdout 70/30:', C_res)

In [ ]:
# Cell 9: Protocol D - session probe  (distinguish the TWO 'nothing' recordings: same label, pure session)
nm_ = y == 'nothing'; Xn = Xz[nm_]; yn = (src[nm_] == 'human_nothing.csv').astype(int)
aucs = []
for trp, tep in StratifiedKFold(5, shuffle=True, random_state=0).split(Xn, yn):
    c = LogisticRegression(max_iter=3000).fit(Xn[trp], yn[trp])
    aucs.append(roc_auc_score(yn[tep], c.decision_function(Xn[tep])))
probe_auc = float(np.mean(aucs))
print(f'D) session probe AUC (nothing-vs-nothing, same class): {probe_auc:.3f}')
print('   AUC ~1.0 => the 104 features encode WHICH recording perfectly (no class info needed).')

In [ ]:
# Cell 10: ALL GRAPHS (dedicated plotting cell) -- 4-CSV dataset only
oof = np.empty(len(y), dtype=object)   # within-session (block) out-of-fold preds for the confusion panel
for k in range(K):
    trm = blk != k; tem = blk == k
    c = LogisticRegression(max_iter=3000).fit(Xz[trm], y[trm]); oof[tem] = c.predict(Xz[tem])
cm = confusion_matrix(y, oof, labels=LABS)

fig, ax = plt.subplots(1, 3, figsize=(21, 6))

# --- Panel 1: accuracy by model x protocol ---
protos = {'A random (leaky)': A_res, 'B block (SNN proto)': B_res, 'C temporal holdout': C_res}
mnames = list(models.keys()); xpos = np.arange(len(mnames)); w = 0.25
for i, (pn, res) in enumerate(protos.items()):
    ax[0].bar(xpos + (i - 1) * w, [res[m][0] for m in mnames], w, label=pn)
ax[0].axhline(SNN_REF, color='k', ls='--', lw=1.2, label=f'SNN ref ({SNN_REF:.3f})')
ax[0].axhline(1/3, color='gray', ls=':', lw=1.2, label='chance (0.33)')
ax[0].set_xticks(xpos); ax[0].set_xticklabels(mnames, rotation=25, ha='right', fontsize=8)
ax[0].set_ylim(0, 1.08); ax[0].set_ylabel('accuracy'); ax[0].legend(fontsize=8, loc='lower left')
ax[0].set_title('Within-real-set accuracy by model & protocol\n(linear == SNN; even a 3-leaf tree ~0.9)')

# --- Panel 2: the confound in one view ---
labels2 = ['logreg\n(B block)', 'tree_d2\n(~3 leaves)', 'chance', 'session probe\nAUC (D)']
vals2 = [B_res['logreg (linear)'][0], B_res['tree_d2 (~3 leaves)'][0], 1/3, probe_auc]
colors2 = ['#1f77b4', '#2ca02c', '#7f7f7f', '#9467bd']
bars = ax[1].bar(range(4), vals2, color=colors2)
for b, v in zip(bars, vals2):
    ax[1].text(b.get_x() + b.get_width()/2, v + 0.02, f'{v:.2f}', ha='center', fontsize=11)
ax[1].set_xticks(range(4)); ax[1].set_xticklabels(labels2, fontsize=8)
ax[1].set_ylim(0, 1.08); ax[1].set_ylabel('accuracy / AUC')
ax[1].set_title('The confound in one view\nany model ~0.98 within-session; session probe AUC 1.0 => session ID')

# --- Panel 3: within-session confusion (logreg, B block OOF) ---
ax[2].imshow(cm, cmap='Blues')
ax[2].set_xticks(range(3)); ax[2].set_yticks(range(3))
ax[2].set_xticklabels(LABS); ax[2].set_yticklabels(LABS)
ax[2].set_xlabel('predicted'); ax[2].set_ylabel('true')
for i in range(3):
    for jx in range(3):
        ax[2].text(jx, i, int(cm[i, jx]), ha='center', va='center',
                   color='white' if cm[i, jx] > cm.max()/2 else 'black', fontsize=11)
ax[2].set_title('Within-session confusion (logreg, B block OOF)\nnear-perfect = trivially separable (the confound)')

plt.tight_layout()
plt.savefig('classical_separability_summary.png', dpi=120); print('saved figure -> classical_separability_summary.png')
plt.show()